# Model Evaluation — Credit Risk XGBoost

This notebook evaluates the trained credit risk model by reproducing the training pipeline from `train_register.py` and generating comprehensive evaluation artifacts:

- ROC curve with AUC annotation
- Precision-Recall curve with Average Precision
- Confusion matrix at optimal threshold
- Classification report (precision, recall, F1)
- Calibration plot (reliability diagram)
- Threshold vs approval rate / default rate tradeoff
- Business interpretation of results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
RANDOM_STATE = 42
THRESHOLD = 0.3

## 1. Load Data & Reproduce Training Pipeline

In [ ]:
DATA_PATH = "../data/complete_feature_dataset.csv"
df = pd.read_csv(DATA_PATH)
print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Target distribution:\n{df['TARGET'].value_counts().to_string()}")
print(f"Default rate: {df['TARGET'].mean():.2%}")

FEATURES = [
    "EXT_SOURCE_3", "EXT_SOURCE_2", "EXT_SOURCE_1", "DAYS_BIRTH",
    "AMT_GOODS_PRICE", "AMT_CREDIT", "AMT_ANNUITY", "DAYS_EMPLOYED",
    "CODE_GENDER", "BUREAU_DEBT_TO_CREDIT_RATIO", "BUREAU_ACTIVE_CREDIT_SUM",
    "NAME_EDUCATION_TYPE", "POS_MEAN_CONTRACT_LENGTH", "PREV_ANNUITY_MEAN",
    "PREV_GOODS_TO_CREDIT_RATIO", "NAME_FAMILY_STATUS", "POS_LATEST_MONTH",
    "ORGANIZATION_TYPE", "BUREAU_AMT_MAX_OVERDUE_EVER", "POS_TOTAL_MONTHS_OBSERVED",
    "AMT_INCOME_TOTAL", "PREV_REFUSAL_RATE", "NAME_INCOME_TYPE", "CNT_CHILDREN",
]

CAT_COLS = ["CODE_GENDER", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
            "ORGANIZATION_TYPE", "NAME_INCOME_TYPE"]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

X = df[FEATURES]
y = df["TARGET"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
class ClipQuantiles(BaseEstimator, TransformerMixin):
    """Clip numeric values at 0.1th and 99.9th percentiles to reduce outlier impact."""
    def __init__(self, qlow=0.001, qhigh=0.999):
        self.qlow, self.qhigh = qlow, qhigh
    def fit(self, X, y=None):
        Xdf = pd.DataFrame(X)
        self.lower_ = Xdf.quantile(self.qlow, interpolation="nearest")
        self.upper_ = Xdf.quantile(self.qhigh, interpolation="nearest")
        return self
    def transform(self, X):
        return pd.DataFrame(X).clip(self.lower_, self.upper_, axis=1).values

# Build the same preprocessing pipeline as train_register.py
prev_ratio_cols = [c for c in NUM_COLS if c.startswith("PREV_") and ("RATIO" in c or "RATE" in c)]
prev_other_cols = [c for c in NUM_COLS if c.startswith("PREV_") and c not in prev_ratio_cols]
other_num_cols  = [c for c in NUM_COLS if not c.startswith("PREV_")]

transformers = []
if other_num_cols:
    transformers.append(("num_other", Pipeline([
        ("clip", ClipQuantiles()), ("impute", SimpleImputer(strategy="median"))
    ]), other_num_cols))
if prev_ratio_cols:
    transformers.append(("num_prev_ratio", Pipeline([
        ("clip", ClipQuantiles()), ("impute", SimpleImputer(strategy="median"))
    ]), prev_ratio_cols))
if prev_other_cols:
    transformers.append(("num_prev_other", Pipeline([
        ("clip", ClipQuantiles()), ("impute", SimpleImputer(strategy="constant", fill_value=0))
    ]), prev_other_cols))
if CAT_COLS:
    transformers.append(("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), CAT_COLS))

pre = ColumnTransformer(transformers, remainder="drop")
clf = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
    n_jobs=-1, random_state=RANDOM_STATE,
)
pipe = Pipeline([("pre", pre), ("clf", clf)])

print("Training pipeline...")
pipe.fit(X_train, y_train)
y_proba = pipe.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"Test AUC: {auc:.4f}")

## 2. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, lw=2, label=f"XGBoost (AUC = {auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Credit Risk Model")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../assets/roc_curve.png", bbox_inches="tight")
plt.show()

## 3. Precision-Recall Curve

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(recall, precision, lw=2, label=f"XGBoost (AP = {ap:.4f})")
baseline = y_test.mean()
ax.axhline(y=baseline, color="k", linestyle="--", lw=1, alpha=0.5,
           label=f"Baseline (prevalence = {baseline:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve — Credit Risk Model")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Confusion Matrix at Threshold = 0.3

In [ ]:
y_pred = (y_proba >= THRESHOLD).astype(int)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=["Repay (0)", "Default (1)"])
disp.plot(ax=ax, cmap="Blues", values_format=",")
ax.set_title(f"Confusion Matrix (threshold = {THRESHOLD})")
plt.tight_layout()
plt.show()

print(f"\nClassification Report (threshold = {THRESHOLD}):")
print(classification_report(y_test, y_pred, target_names=["Repay", "Default"], digits=4))

## 5. Calibration Plot (Reliability Diagram)

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10, strategy="uniform")

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(prob_pred, prob_true, "s-", lw=2, label="XGBoost")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Perfectly calibrated")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Plot (Reliability Diagram)")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Threshold vs Approval Rate / Default Rate Tradeoff

This is the key business chart: as we lower the decision threshold, we approve more applicants but also accept more defaults.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 50)
approval_rates = []
default_rates_among_approved = []

for t in thresholds:
    approved = y_proba < t  # approve if predicted default prob < threshold
    approval_rate = approved.mean()
    if approved.sum() > 0:
        default_rate = y_test.values[approved].mean()
    else:
        default_rate = 0.0
    approval_rates.append(approval_rate)
    default_rates_among_approved.append(default_rate)

fig, ax1 = plt.subplots(figsize=(9, 6))
color1, color2 = "#2196F3", "#F44336"

ax1.plot(thresholds, np.array(approval_rates) * 100, color=color1, lw=2, label="Approval rate")
ax1.set_xlabel("Decision Threshold (reject if P(default) >= threshold)")
ax1.set_ylabel("Approval Rate (%)", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

ax2 = ax1.twinx()
ax2.plot(thresholds, np.array(default_rates_among_approved) * 100, color=color2, lw=2, label="Default rate (among approved)")
ax2.set_ylabel("Default Rate Among Approved (%)", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

# Mark current threshold
ax1.axvline(x=THRESHOLD, color="gray", linestyle="--", alpha=0.7, label=f"Current threshold ({THRESHOLD})")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

ax1.set_title("Threshold vs Approval Rate / Default Rate Tradeoff")
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Business Interpretation

**Key takeaways:**

- **AUC ~0.77** demonstrates the model meaningfully separates defaulters from non-defaulters across the full range of thresholds. For a credit risk screening model with 8.1% base default rate, this provides substantial lift over random.

- **Threshold = 0.3** is deliberately aggressive (low) to prioritize **recall of defaults** — in lending, missing a default (false negative) is far more costly than rejecting a good applicant (false positive). The classification report above shows the precision/recall tradeoff at this operating point.

- **Calibration** shows whether predicted probabilities match observed default rates. If the calibration curve lies above the diagonal, the model underestimates risk; below means it overestimates. Either way, the threshold-based decision remains valid, but calibrated probabilities matter for risk pricing and capital allocation.

- **Approval rate vs default rate chart** is the most business-relevant: it shows the direct tradeoff a lending team faces. Moving the threshold left rejects more applicants but reduces losses; moving right grows the book but increases defaults.